# Fine-tuning YOLO11n sur American Sign Language Letters

**Plateforme :** Google Colab (Runtime > Change runtime type > T4 GPU)

**Objectif :** Entraîner un modèle YOLO11n sur le dataset ASL (26 lettres A-Z) pour la détection en temps réel.

**Durée estimée :** ~1h30-2h sur T4.

## 1. Vérification GPU

In [ ]:
!nvidia-smi

## 2. Installation des dépendances

In [ ]:
!pip install -q ultralytics==8.3.* roboflow supervision

## 3. Téléchargement du dataset ASL via Roboflow

Dataset public : **American Sign Language Letters** par David Lee (26 classes, ~1700 images, déjà splité train/val/test).

In [ ]:
import os
from roboflow import Roboflow

API_KEY = os.environ.get("ROBOFLOW_API_KEY", "01ppKST0VVzBcfnAElsw")

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("david-lee-d0rhs").project("american-sign-language-letters")
version = project.version(6)
dataset = version.download("yolov11")

print("Dataset téléchargé dans:", dataset.location)

In [ ]:
# Vérifier le data.yaml
import yaml
from pathlib import Path

data_yaml = Path(dataset.location) / "data.yaml"
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)
print(cfg)
print(f"\nNombre de classes: {cfg['nc']}")
print(f"Classes: {cfg['names']}")

## 4. Entraînement YOLO11n

- 50 epochs, image size 640, batch 32
- Patience 15 (early stopping)
- Sauvegarde tous les 10 epochs
- Augmentations par défaut d'Ultralytics (mosaic, mixup, hsv, flips)

In [ ]:
from ultralytics import YOLO
import torch

torch.manual_seed(42)

model = YOLO('yolo11n.pt')

results = model.train(
    data=str(data_yaml),
    epochs=50,
    imgsz=640,
    batch=32,
    patience=15,
    save_period=10,
    project='runs/detect',
    name='yolo11n_asl',
    seed=42,
    plots=True,
    verbose=True,
)

## 5. Évaluation sur le test set

In [ ]:
best_model = YOLO('runs/detect/yolo11n_asl/weights/best.pt')
metrics = best_model.val(data=str(data_yaml), split='test', plots=True)
print(f"\n=== Résultats sur test set ===")
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")

## 6. Mesure du temps d'inférence

In [ ]:
import time
import glob
from PIL import Image

test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")[:50]
print(f"Mesure sur {len(test_images)} images")

_ = best_model.predict(test_images[0], verbose=False)  # warmup

start = time.time()
for img in test_images:
    _ = best_model.predict(img, verbose=False)
elapsed = time.time() - start
print(f"Temps moyen par image: {1000*elapsed/len(test_images):.2f} ms")
print(f"FPS estimé: {len(test_images)/elapsed:.1f}")

## 7. Sauvegarde et téléchargement du modèle

In [ ]:
import shutil, os
from google.colab import files

# Taille modèle
size_mb = os.path.getsize('runs/detect/yolo11n_asl/weights/best.pt') / 1024 / 1024
print(f"Taille modèle best.pt: {size_mb:.2f} MB")

# Archive du run complet (weights + plots + résultats CSV)
shutil.make_archive('yolo11n_asl_run', 'zip', 'runs/detect/yolo11n_asl')
print("Archive créée: yolo11n_asl_run.zip")

# Téléchargement
files.download('runs/detect/yolo11n_asl/weights/best.pt')
files.download('yolo11n_asl_run.zip')